In [1]:
# Import Dataset For MINST
import pandas as pd
import numpy as np
# SRC: https://www.kaggle.com/code/wwsalmon/simple-mnist-nn-from-scratch-numpy-no-tf-keras

In [13]:
%cd sample_data
%ls 

/content/sample_data
anscombe.json*                mnist_test.csv
california_housing_test.csv   mnist_train_small.csv
california_housing_train.csv  README.md*


In [2]:
## Explore The Data
data = pd.read_csv("./mnist_train.csv")
data.head(3)

,label,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [45]:
data = np.array(data)
m, n = data.shape # get number of rows and number of element in 2D array
# np.random.shuffle(data) # WHY THIS

data_dev = data[0: 1000].T
Y_dev = data_dev[0]
X_dev = data_dev[1: m]

data_train = data[1000: m].T
X_train = data_train[1: n]
Y_train = data_train[0]

In [46]:
X_train.shape # (NLine, NColumn)

(784, 9000)

In [47]:
X_train

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [ ]:
# Init The Hyper Parameter
nbr_neural_hidden_layer = 10
def init_param():
    # n is number of columns in dataset
    W_1 = np.random.rand(10, 784) - 0.5
    B_1 = np.random.rand(10, 1) - 0.5

    W_2 = np.random.rand(10, 10) - 0.5
    B_2 = np.random.rand(10, 1) - 0.5
    return W_1, B_1, W_2, B_2
init_param()

# ForWard Propagation
def ReLU(Z):
    return np.maximum(Z, 0)

def softMaX(Z):
    # Subtracting the max for numerical stability
    exp_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

def forward_prop(W_1, B_1, W_2, B_2, X):
    # Calculate Z_1 = W_1 * A_0 + B_1
    Z_1 = W_1.dot(X) + B_1
    A_1 = ReLU(Z_1)
    
    # Calculate Z_2 = W_2 * A_1 + B_2
    Z_2 = W_2.dot(A_1) + B_2
    A_2 = softMaX(Z_2)
    return Z_1, A_1, Z_2, A_2

# BackWard Propagation
def on_hot_code(Y):
    Y_len = len(Y)
    new_Y = np.zeros(shape=(10, Y_len), dtype=np.uint8)
    
    for i in range(Y_len):
        new_Y[Y[i]][i] = 1
    return new_Y

def ReLU_deriv(Z):
    return Z > 0 # Element Big Then 0 Converted TO 1 ELSE TO 0

def backward_prop(Z_1, A_1, Z_2, A_2, W_1, W_2, X, Y):
    Y_new = on_hot_code(Y)
    ### FOR OUTPUR LAYER
    dZ_2 = A_2 - Y_new
    dW_2 = (1/m) * dZ_2.dot(A_1.T)
    dB_2 = (1 / m) * np.sum(dZ_2, axis=1, keepdims=True)

    ### FOR HIDDEN LAYER
    dZ_1 = W_2.T.dot(dZ_2) * ReLU_deriv(Z_1)
    dW_1 = (1/m) * dZ_1.dot(X.T) # where X is A_0
    dB_1 = (1 / m) * np.sum(dZ_1, axis=1, keepdims=True)
    
    return dW_2, dB_2, dW_1, dB_1


# Update Params
def update_params(W_1, W_2, B_1, B_2, dW_1, dW_2, dB_1, dB_2, alpha):
    W_2 = W_2 - alpha * dW_2
    B_2 = B_2 - alpha * dB_2
    W_1 = W_1 - alpha * dW_1
    B_1 = B_1 - alpha * dB_1

    return W_2, B_2, W_1, B_1


In [50]:
# Model Predict A_2 = [0.01, 0.02, 0.85, 0.02, ...]
# Means Model Predict 2 Because The High Value is 0.85
def get_prediction(A_2):
    return np.argmax(A_2, axis=0)

def get_accuracy(predictions, Y):
    return np.sum(Y == predictions) / Y.size

def gradient_descent(X, Y, alpha, epochs):
    W_1, B_1, W_2, B_2 = init_param()
    # print(new_Y)
    for i in range(epochs):
        Z_1, A_1, Z_2, A_2 = forward_prop(W_1, B_1, W_2, B_2, X)
        dW_2, dB_2, dW_1, dB_1 = backward_prop(Z_1, A_1, Z_2, A_2, W_1, W_2, X, Y)
        W_2, B_2, W_1, B_1 = update_params(W_1, W_2, B_1, B_2, dW_1, dW_2, dB_1, dB_2, alpha)
        if (i % 1 == 0):
            # Get Prediction and Accuray
            print("Iteration: ", i)
            predicted_value = get_prediction(A_2=A_2)
            print("Predicted Value is: ", predicted_value)
            
            accuracy = get_accuracy(predictions=predicted_value, Y=Y)
            print("Accuracy is: ", accuracy)
    return W_2, W_1, B_1, B_2

In [51]:
W_2, W_1, B_1, B_2 = gradient_descent(X=X_train, Y=Y_train, alpha=0.1, epochs=100)

Iteration:  0
Predicted Value is:  [3 3 3 ... 3 3 3]
Accuracy is:  0.06655555555555556
Iteration:  1
Predicted Value is:  [9 4 1 ... 9 9 1]
Accuracy is:  0.10322222222222223
Iteration:  2
Predicted Value is:  [0 0 0 ... 0 0 0]
Accuracy is:  0.09933333333333333
Iteration:  3
Predicted Value is:  [0 0 0 ... 0 0 0]
Accuracy is:  0.09944444444444445
Iteration:  4
Predicted Value is:  [0 0 0 ... 0 0 0]
Accuracy is:  0.09944444444444445
Iteration:  5
Predicted Value is:  [0 0 0 ... 0 0 0]
Accuracy is:  0.09944444444444445
Iteration:  6
Predicted Value is:  [0 0 0 ... 0 0 0]
Accuracy is:  0.09944444444444445
Iteration:  7
Predicted Value is:  [0 0 0 ... 0 0 0]
Accuracy is:  0.09944444444444445
Iteration:  8
Predicted Value is:  [0 0 0 ... 0 0 0]
Accuracy is:  0.09944444444444445
Iteration:  9
Predicted Value is:  [0 0 0 ... 0 0 0]
Accuracy is:  0.09944444444444445
Iteration:  10
Predicted Value is:  [0 0 0 ... 0 0 0]
Accuracy is:  0.09944444444444445
Iteration:  11
Predicted Value is:  [0 0 0